In [ ]:
import os
from functools import lru_cache
from typing import List, Dict

# ========== 第1步：配置（本地加载不需要HF_ENDPOINT，但保留以防其他依赖需要） ==========
os.environ.update({
    "no_proxy": "localhost,127.0.0.1,::1"
})

from functools import lru_cache
from typing import List, Dict
import gradio as gr
from transformers import pipeline
import torch
import gc

# ========== 第2步：配置本地模型路径 ==========
# 方式A：绝对路径（推荐，最稳妥）
# LOCAL_MODEL_PATH = r"E:\Models\Qwen2.5-0.5B-Instruct"  # ← 改成你的实际路径

# 方式B：相对路径（如果模型文件夹和代码在同一目录下）
LOCAL_MODEL_PATH = "./models/Qwen2.5-0.5B-Instruct"  

# 方式C：从环境变量读取（适合多机部署）
# LOCAL_MODEL_PATH = os.environ.get("LOCAL_MODEL_PATH", "./Qwen2.5-0.5B-Instruct")

print(f"📂 将从本地加载模型: {LOCAL_MODEL_PATH}")
print(f"📂 路径是否存在: {os.path.exists(LOCAL_MODEL_PATH)}")

@lru_cache(maxsize=1)
def get_pipe():
    """🏭 单例工厂：从本地加载模型"""
    print(f"🚀 正在从本地加载模型: {LOCAL_MODEL_PATH}")
    
    # 检查路径是否存在（防止报错）
    if not os.path.exists(LOCAL_MODEL_PATH):
        raise FileNotFoundError(
            f"❌ 模型路径不存在: {LOCAL_MODEL_PATH}\n"
            f"请确保路径正确，或先运行 snapshot_download 下载模型到本地"
        )
    
    # 方式1：直接传本地路径给 pipeline
    pipe = pipeline(
        "text-generation",
        model=LOCAL_MODEL_PATH,        
        tokenizer=LOCAL_MODEL_PATH,    
        device="cuda",
        torch_dtype=torch.float16,
        trust_remote_code=True         # ← 本地模型通常需要这个（允许执行模型目录下的自定义代码）
    )
    
    print("✅ 本地模型加载完成！")
    return pipe

def clean(content) -> str:
    """🧼 万能清洗"""
    if isinstance(content, (list, tuple)) and content:
        return str(content[0])
    return str(content) if content else ""

def build_messages(history: List, current_msg: str) -> List[Dict]:
    """📦 历史记录组装工厂"""
    messages = []
    for turn in history or []:
        if isinstance(turn, dict):
            messages.append({"role": turn.get("role", "user"), "content": clean(turn.get("content"))})
        elif isinstance(turn, (list, tuple)) and len(turn) >= 2:
            messages.extend([
                {"role": "user", "content": clean(turn[0])},
                {"role": "assistant", "content": clean(turn[1])}
            ])
    messages.append({"role": "user", "content": clean(current_msg)})
    return messages

def chat(message: str, history: List) -> str:
    torch.cuda.empty_cache()
    gc.collect()
    
    try:
        outputs = get_pipe()(
            build_messages(history, message),
            max_new_tokens=256,
            pad_token_id=get_pipe().tokenizer.eos_token_id,
            do_sample=True,
            temperature=0.7
        )
        return outputs[0]["generated_text"][-1]["content"]
    
    except Exception as e:
        print(f"❌ Error: {e}")
        return f"出错了: {e}"

if __name__ == "__main__":
    # 启动时检查一次模型是否存在
    if not os.path.exists(LOCAL_MODEL_PATH):
        print(f"⚠️ 警告: 模型路径 {LOCAL_MODEL_PATH} 不存在！")
        print(f"请先将模型下载到该路径，或修改 LOCAL_MODEL_PATH 变量")
    
    gr.ChatInterface(
        fn=chat,
        title="Qwen2.5-0.5B (本地版)",
        description="✨ 模型从本地加载，无需网络连接"
    ).launch()